# 03 — Phase 1 protocol review

This notebook audits the protocol-aligned serializer on train/validation only. It does not download a model, generate embeddings, evaluate the test split, or display a real patient history.

## Protocol and unresolved decisions

In [ ]:
import datetime as dt
import json
from pathlib import Path

from IPython.display import display

from temporal_ehr.config import resolve_path
from temporal_ehr.data.task_registry import load_task_registry
from temporal_ehr.experiments import load_experiment_panel
from temporal_ehr.serialization.blocks import ClinicalEvent, build_prediction_windows, deduplicate_events
from temporal_ehr.serialization.conditions import build_full_history_conditions
from temporal_ehr.serialization.packing import load_qwen_tokenizer
from temporal_ehr.serialization.qc import run_protocol_alignment_qc
from temporal_ehr.serialization.render import render_condition

panel_paths = {
    name: Path(f"configs/experiments/{name}.yaml")
    for name in ("pilot_three_task", "proposed_four_task", "locked_panel")
}
panel_review = {name: load_experiment_panel(path) for name, path in panel_paths.items()}
display({name: {"status": value["status"], "tasks": value["tasks"]} for name, value in panel_review.items()})

## Entirely synthetic C0-C3 demonstration

Only invented events are rendered. `T` is the invented prediction time; no calendar date appears in model text.

In [ ]:
synthetic_t = dt.datetime(2030, 1, 1, 12)
synthetic_source = (
    ClinicalEvent(synthetic_t - dt.timedelta(days=240), "SYNTHETIC/ALPHA"),
    ClinicalEvent(synthetic_t - dt.timedelta(days=125), "SYNTHETIC/RECURRENT"),
    ClinicalEvent(synthetic_t - dt.timedelta(days=61), "SYNTHETIC/RECURRENT"),
    ClinicalEvent(synthetic_t - dt.timedelta(days=10), "SYNTHETIC/OMEGA"),
)
synthetic_events, synthetic_duplicates = deduplicate_events(synthetic_source)
synthetic_windows = build_prediction_windows(synthetic_events, synthetic_t, window_days=30)
synthetic_conditions = build_full_history_conditions(
    synthetic_windows, seed=20260815, instance_key="entirely-synthetic"
)
display(
    {
        name: render_condition(condition, {}) if condition.eligible else condition.ineligibility_reason
        for name, condition in synthetic_conditions.items()
    }
)
assert synthetic_duplicates == 0

## Governed aggregate review configuration

Real histories are streamed into aggregate counters and private hash-only manifests. They are never displayed.

In [ ]:
event_path = resolve_path(None, "EHRSHOT_DATA_PATH", must_be_file=True)
asset_root = resolve_path(None, "EHRSHOT_ASSET_ROOT", must_be_dir=True)
cohort_path = resolve_path(None, "EHRSHOT_COHORT_PATH", must_be_file=True)
qwen_path = resolve_path(None, "QWEN_EMBEDDING_PATH", must_be_dir=True)
registry = load_task_registry(Path("configs/tasks.yaml"))
tasks = tuple(registry["primary_panel"])
tokenizer = load_qwen_tokenizer(qwen_path)
display(
    {
        "panel": "phase1_three_task_pilot",
        "tasks": tasks,
        "splits": ["train", "validation"],
        "token_budgets": registry["serialization"]["token_budgets"],
        "c3_seeds": registry["serialization"]["c3_perturbation_seeds"],
        "local_tokenizer_directory": qwen_path.name,
    }
)

## Duplicate removal and prediction-relative windows

In [ ]:
review = run_protocol_alignment_qc(
    event_path,
    cohort_path,
    asset_root,
    tokenizer=tokenizer,
    tokenizer_root=qwen_path,
    registry=registry,
    tasks=tasks,
    splits=("train", "validation"),
    token_budgets=tuple(registry["serialization"]["token_budgets"]),
    c3_seeds=tuple(registry["serialization"]["c3_perturbation_seeds"]),
    minimum_group_size=int(registry["serialization"]["minimum_report_group_size"]),
)
display(
    {
        "duplicate_removal": review["duplicate_removal"],
        "prediction_windows": review["prediction_window_distributions"],
    }
)

## Temporal-condition eligibility

In [ ]:
display(
    {
        "eligibility": review["condition_eligibility"],
        "ineligibility_reason_counts": review["ineligibility_reason_counts"],
    }
)

## Packing and information-reduction baselines

In [ ]:
display(
    {
        "content_matched_primary": review["content_matched_primary"],
        "independent_operational": review["independent_operational"],
        "information_reduction_baselines": review["information_reduction_baselines"],
    }
)

## Machine-readable Phase 3 checklist

In [ ]:
print(json.dumps({"protocol_decision": review["protocol_decision"], **review["checklist"]}, sort_keys=True))